<a href="https://colab.research.google.com/github/SaharaAli16/UNT_Teaching/blob/main/DTSC_4050/Week_12_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DTSC 4050 — In-Class Activity: Simple Linear Regression

This activity follows the regression topics in the lecture slides.

## Learning goals
By the end of class, you should be able to:
- identify predictor **X** and response **Y**;
- inspect a scatter plot before fitting a model;
- calculate and interpret correlation;
- fit and interpret a simple linear regression;
- distinguish observed values, fitted values, and residuals;
- calculate MAE, MSE, and RMSE;
- use residual plots to assess a linear model;
- make predictions and recognize extrapolation;
- bootstrap a regression slope and use its 95% confidence interval to assess whether a true slope of 0 is plausible.

Complete all **YOUR TURN** sections.


### Additional learning outcomes
- Build and validate **linear, multiple, and polynomial regression models** in Python with **pandas** and **scikit-learn**, documenting code and assumptions.
- Evaluate and interpret model performance using **R², adjusted R², and MSE**, and justify model selection based on these metrics and diagnostic checks such as **residual analysis**.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(4050)


## 2. Warm-up: Which Variable Is X and Which Is Y?

We begin with the baby example from the lesson.

| Baby | Gestational Days (X) | Birth Weight, oz (Y) |
|---|---:|---:|
| 1 | 270 | 112 |
| 2 | 278 | 118 |
| 3 | 285 | 124 |
| 4 | 292 | 126 |
| 5 | 300 | 132 |

Discuss:
1. Which variable is the predictor?
2. Which variable is the response?



In [ ]:
babies = pd.DataFrame({
    "gestational_days": [270, 278, 285, 292, 300],
    "birth_weight_oz": [112, 118, 124, 126, 132]
})
babies


## 3. Look Before You Infer

Start with a scatter plot. Look for a roughly linear pattern, unusual observations, and curvature.


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(babies["gestational_days"], babies["birth_weight_oz"])
plt.xlabel("Gestational Days")
plt.ylabel("Birth Weight (oz)")
plt.title("Gestational Days vs. Birth Weight")
plt.show()


### YOUR TURN 1
**A.** Is the association positive, negative, or approximately zero?  
**Answer:**

**B.** Does a straight-line model appear reasonable? Explain briefly.  
**Answer:**


## 4. Correlation

The correlation coefficient **r** summarizes the direction and strength of a **linear** association.

- $r > 0$: positive linear association
- $r < 0$: negative linear association
- $r$ near 0: weak or no linear association
- correlation has no units
- correlation describes association, not causation


In [ ]:
r = babies["gestational_days"].corr(babies["birth_weight_oz"])
print(f"Correlation r = {r:.3f}")


## 5. Linear Regression with `scikit-learn`

Python's **scikit-learn (`sklearn`)** package provides a ready-to-use `LinearRegression` model.

We first import it with:

```python
from sklearn.linear_model import LinearRegression
```

The fitted simple linear regression equation is

$$\hat{y} = b_0 + b_1x$$

where:
- $b_0$ is the **intercept**;
- $b_1$ is the **slope/coefficient**;
- $x$ is the predictor;
- $\hat{y}$ is the fitted or predicted response.

The basic sklearn workflow is:

1. Separate the **features (X)** and **target (y)**.
2. Create the model: `LinearRegression()`.
3. Train it: `model.fit(X, y)`.
4. Inspect `model.intercept_` and `model.coef_`.
5. Make predictions with `model.predict(...)`.


In [ ]:
# Import the LinearRegression class
from sklearn.linear_model import LinearRegression

# X must be a 2-dimensional table of features
X = babies[["gestational_days"]]

# y is the response/target
y = babies["birth_weight_oz"]

# Step 1: Create a Linear Regression model
model = LinearRegression()

# Step 2: Fit/train the model using the observed data
model.fit(X, y)

# Step 3: Examine the fitted parameters
intercept = model.intercept_
slope = model.coef_[0]

print(f"Intercept = {intercept:.3f}")
print(f"Slope = {slope:.3f} oz/day")
print(f"Equation: predicted weight = {intercept:.3f} + {slope:.3f} × gestational days")


### YOUR TURN 2 — Interpret the Model
**A. Slope:** For each additional gestational day, fitted birth weight changes by approximately ______ ounces.

**B. Intercept:** What does the intercept mathematically represent?

**Answer:**


## 5.1 Understanding the `sklearn` Workflow

A useful way to remember the process is:

**Create → Fit → Inspect → Predict**

`LinearRegression()` creates an untrained model.  
`fit(X, y)` estimates the intercept and coefficient from the training data.  
`predict(X)` uses the fitted equation to calculate predicted values.

### Important: Why is X written with double brackets?

Scikit-learn expects the feature matrix `X` to be **2-dimensional**, even when we have only one predictor.

- `babies["gestational_days"]` → one-dimensional Series
- `babies[["gestational_days"]]` → two-dimensional DataFrame

For this activity, use the second form for `X`.


In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

print("\nFirst few X values:")
display(X.head())

print("\nFirst few y values:")
display(y.head())


### YOUR TURN — Build an `sklearn` Model Yourself

Complete the missing lines below. Your model should use **gestational days** to predict **birth weight**.

After fitting the model, print its intercept and slope.


In [ ]:
# 1. Select predictor and response
X_practice = babies[[_____________________]]
y_practice = babies[_____________________]

# 2. Create the model
practice_model = _________________________

# 3. Fit the model
practice_model.___________________________

# 4. Display the fitted parameters
print("Intercept:", _______________________)
print("Slope:", ___________________________)


## 6. Visualize the Fitted Regression Line

In [ ]:
x_line = np.linspace(babies["gestational_days"].min(),
                     babies["gestational_days"].max(), 100)
line_df = pd.DataFrame({"gestational_days": x_line})
y_line = model.predict(line_df)

plt.figure(figsize=(7, 5))
plt.scatter(babies["gestational_days"], babies["birth_weight_oz"], label="Observed data")
plt.plot(x_line, y_line, label="Fitted line")
plt.xlabel("Gestational Days")
plt.ylabel("Birth Weight (oz)")
plt.title("Simple Linear Regression")
plt.legend()
plt.show()


## 7. Observed Values, Fitted Values, and Residuals

$$\text{Residual} = y - \hat{y}$$

A positive residual means the observation is above the fitted line. A negative residual means it is below the line.


In [ ]:
babies["predicted_weight"] = model.predict(X)
babies["residual"] = babies["birth_weight_oz"] - babies["predicted_weight"]
babies.round(2)


### YOUR TURN 3
1. Which baby has the largest positive residual?
2. Which has the largest negative residual?


**Answer:**


## 8. Prediction Error Metrics

- **MAE:** average of $|observed-predicted|$
- **MSE:** average squared residual
- **RMSE:** square root of MSE; same units as Y


In [ ]:
mae = mean_absolute_error(y, babies["predicted_weight"])
mse = mean_squared_error(y, babies["predicted_weight"])
rmse = np.sqrt(mse)

print(f"MAE  = {mae:.3f} oz")
print(f"MSE  = {mse:.3f} squared oz")
print(f"RMSE = {rmse:.3f} oz")


### `model.score()` and $R^2$

For linear regression, scikit-learn's `model.score(X, y)` returns the **coefficient of determination**, $R^2$.

$R^2$ describes how much of the variation in the response is explained by the fitted model **for the data being scored**.

- closer to 1 → the fitted model explains more of the variation;
- near 0 → it explains little more variation than predicting the mean;
- it should be interpreted together with the scatter plot, residuals, and the context of the problem.

For this tiny five-observation example, treat the value as a demonstration of the sklearn method rather than as evidence of strong out-of-sample performance.


In [ ]:
r_squared = model.score(X, y)
print(f"R² = {r_squared:.3f}")


### YOUR TURN 4
Which error metrics above are expressed in the same units as birth weight?

**Answer:**


## 9. Examine the Residual Plot

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(babies["gestational_days"], babies["residual"])
plt.axhline(0, linestyle="--")
plt.xlabel("Gestational Days")
plt.ylabel("Residual (oz)")
plt.title("Residual Plot")
plt.show()


## 10. Prediction with `sklearn`

Once a `LinearRegression` model has been fitted, use:

```python
model.predict(new_X)
```

The new predictor values must have the same feature structure used when the model was trained.

Below, we ask the fitted sklearn model to predict birth weight at 300 gestational days.


In [ ]:
new_x = pd.DataFrame({"gestational_days": [300]})
prediction = model.predict(new_x)[0]
print(f"Predicted birth weight at 300 days = {prediction:.2f} oz")


### YOUR TURN 5
Modify the code below to predict birth weight at **290 gestational days**.

In [ ]:
# YOUR CODE HERE
new_days = 290

# Hint:
# new_value = pd.DataFrame({"gestational_days": [new_days]})
# ...


## 11. Extrapolation

The observed X values range from 270 to 300 days. A prediction far outside that range is **extrapolation**.


In [ ]:
outside_x = pd.DataFrame({"gestational_days": [400]})
outside_prediction = model.predict(outside_x)[0]
print(f"Mathematical model prediction at 400 days = {outside_prediction:.2f} oz")


# Part II — Inference for the Regression Slope

The sample line is an estimate of an unknown population relationship. If we had collected a different random sample, our estimated slope would usually differ.

We will use bootstrap resampling to examine this uncertainty.


## 12. A Larger Synthetic Sample

In [ ]:
n = 80
gest_days = np.random.normal(280, 10, n)
birth_weight = 0.55 * gest_days - 35 + np.random.normal(0, 8, n)

sample = pd.DataFrame({
    "gestational_days": gest_days,
    "birth_weight_oz": birth_weight
})

sample.head()


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(sample["gestational_days"], sample["birth_weight_oz"])
plt.xlabel("Gestational Days")
plt.ylabel("Birth Weight (oz)")
plt.title("Larger Sample")
plt.show()


## 13. Estimate the Sample Slope

In [ ]:
sample_model = LinearRegression()
sample_model.fit(sample[["gestational_days"]], sample["birth_weight_oz"])

observed_slope = sample_model.coef_[0]
print(f"Observed sample slope = {observed_slope:.3f} oz/day")


## 14. Bootstrap the Slope

Each bootstrap repetition:
1. resamples rows **with replacement**;
2. fits a regression line;
3. saves the slope.


In [ ]:
bootstrap_slopes = []

for i in range(1000):
    boot = sample.sample(n=len(sample), replace=True)
    boot_model = LinearRegression()
    boot_model.fit(boot[["gestational_days"]], boot["birth_weight_oz"])
    bootstrap_slopes.append(boot_model.coef_[0])

bootstrap_slopes = np.array(bootstrap_slopes)

plt.figure(figsize=(8, 5))
plt.hist(bootstrap_slopes, bins=30)
plt.axvline(observed_slope, linestyle="--", label="Observed slope")
plt.xlabel("Bootstrap Slope")
plt.ylabel("Frequency")
plt.title("Bootstrap Distribution of Slopes")
plt.legend()
plt.show()


## 15. Construct an Approximate 95% Bootstrap Confidence Interval

In [ ]:
lower, upper = np.percentile(bootstrap_slopes, [2.5, 97.5])

print(f"Observed slope = {observed_slope:.3f}")
print(f"Approximate 95% bootstrap CI = ({lower:.3f}, {upper:.3f})")


# When a Straight Line Is Not Enough

A correlation near zero does not necessarily mean that two variables have no relationship. The relationship may simply be non-linear.


In [ ]:
x_curve = np.linspace(-5, 5, 100)
y_curve = x_curve**2 + np.random.normal(0, 2, 100)
curved = pd.DataFrame({"X": x_curve, "Y": y_curve})

print(f"Correlation = {curved['X'].corr(curved['Y']):.3f}")

plt.figure(figsize=(7, 5))
plt.scatter(curved["X"], curved["Y"])
plt.xlabel("X")
plt.ylabel("Y")
plt.title("A Curved Relationship")
plt.show()


### YOUR TURN 6
1. Is there a visible relationship between X and Y?
2. Would one straight line summarize it well?


**Answer:**


# 16. Multiple & Polynomial Regression and Model Selection

We will now extend simple linear regression to compare three models:

1. **Simple linear regression** — one predictor.
2. **Multiple linear regression** — two or more predictors.
3. **Polynomial regression** — polynomial terms allow a curved relationship.

For every model, document the predictors, response, assumptions, performance metrics, and residual diagnostics.


## Performance Metrics

We will compare models using:

- **R²:** proportion of variation in Y explained by the fitted model.
- **Adjusted R²:** adjusts R² for the number of predictors; adding unnecessary predictors can reduce it.
- **MSE:** mean squared prediction error; smaller values indicate smaller squared errors.

Adjusted R² is calculated as

$$R^2_{adj}=1-(1-R^2)\frac{n-1}{n-p-1}$$

where $n$ is the number of observations and $p$ is the number of predictor features.

**Important:** Do not choose a model from one metric alone. Also inspect the residuals and consider whether the model form makes sense.


In [ ]:
def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)


## Create a Dataset

We will predict **exam score** from **study hours** and **attendance**. The generated relationship also contains some curvature in study hours.


In [ ]:
rng = np.random.default_rng(4050)
n = 100

regression_data = pd.DataFrame({
    "study_hours": rng.uniform(1, 12, n),
    "attendance": rng.uniform(60, 100, n)
})

regression_data["exam_score"] = (
    35
    + 3.0 * regression_data["study_hours"]
    + 0.22 * regression_data["attendance"]
    - 0.12 * regression_data["study_hours"]**2
    + rng.normal(0, 4, n)
)

regression_data.head()


## Model 1 — Simple Linear Regression

In [ ]:
X_simple = regression_data[["study_hours"]]
y = regression_data["exam_score"]

simple_model = LinearRegression()
simple_model.fit(X_simple, y)
simple_pred = simple_model.predict(X_simple)

simple_r2 = simple_model.score(X_simple, y)
simple_adj = adjusted_r2(simple_r2, len(y), X_simple.shape[1])
simple_mse = mean_squared_error(y, simple_pred)

print(f"R² = {simple_r2:.3f}")
print(f"Adjusted R² = {simple_adj:.3f}")
print(f"MSE = {simple_mse:.3f}")


In [ ]:
simple_residuals = y - simple_pred

plt.figure(figsize=(7,5))
plt.scatter(simple_pred, simple_residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Exam Score")
plt.ylabel("Residual")
plt.title("Residual Plot — Simple Linear Regression")
plt.show()


## Model 2 — Multiple Linear Regression

In [ ]:
X_multiple = regression_data[["study_hours", "attendance"]]

multiple_model = LinearRegression()
multiple_model.fit(X_multiple, y)
multiple_pred = multiple_model.predict(X_multiple)

multiple_r2 = multiple_model.score(X_multiple, y)
multiple_adj = adjusted_r2(multiple_r2, len(y), X_multiple.shape[1])
multiple_mse = mean_squared_error(y, multiple_pred)

print(f"R² = {multiple_r2:.3f}")
print(f"Adjusted R² = {multiple_adj:.3f}")
print(f"MSE = {multiple_mse:.3f}")


In [ ]:
multiple_residuals = y - multiple_pred

plt.figure(figsize=(7,5))
plt.scatter(multiple_pred, multiple_residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Exam Score")
plt.ylabel("Residual")
plt.title("Residual Plot — Multiple Linear Regression")
plt.show()


## Model 3 — Polynomial Regression

Scikit-learn's `PolynomialFeatures` can create terms such as `study_hours²`. We then fit `LinearRegression` using the transformed features.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
study_poly = poly.fit_transform(regression_data[["study_hours"]])

study_poly_df = pd.DataFrame(
    study_poly,
    columns=poly.get_feature_names_out(["study_hours"]),
    index=regression_data.index
)

X_poly = pd.concat(
    [study_poly_df, regression_data[["attendance"]]],
    axis=1
)

poly_model = LinearRegression()
poly_model.fit(X_poly, y)
poly_pred = poly_model.predict(X_poly)

poly_r2 = poly_model.score(X_poly, y)
poly_adj = adjusted_r2(poly_r2, len(y), X_poly.shape[1])
poly_mse = mean_squared_error(y, poly_pred)

print(f"R² = {poly_r2:.3f}")
print(f"Adjusted R² = {poly_adj:.3f}")
print(f"MSE = {poly_mse:.3f}")


In [ ]:
poly_residuals = y - poly_pred

plt.figure(figsize=(7,5))
plt.scatter(poly_pred, poly_residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Exam Score")
plt.ylabel("Residual")
plt.title("Residual Plot — Polynomial Regression")
plt.show()


## Compare the Three Models

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Simple Linear", "Multiple Linear", "Polynomial"],
    "R²": [simple_r2, multiple_r2, poly_r2],
    "Adjusted R²": [simple_adj, multiple_adj, poly_adj],
    "MSE": [simple_mse, multiple_mse, poly_mse]
})

comparison.round(3)


## Validate a Model with Training and Test Data

A model should also be evaluated on observations that were **not used to fit it**. Below we validate the polynomial model with a train/test split.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y,
    test_size=0.25,
    random_state=4050
)

validated_model = LinearRegression()
validated_model.fit(X_train, y_train)

train_pred = validated_model.predict(X_train)
test_pred = validated_model.predict(X_test)

print(f"Training R² = {validated_model.score(X_train, y_train):.3f}")
print(f"Test R² = {validated_model.score(X_test, y_test):.3f}")
print(f"Training MSE = {mean_squared_error(y_train, train_pred):.3f}")
print(f"Test MSE = {mean_squared_error(y_test, test_pred):.3f}")
